# Tutorial 15 — Fine-tuning ChemBERTa for Toxicity Prediction
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

ChemBERTa is a BERT-style transformer pre-trained on 77M SMILES strings from PubChem. It learns chemical language representations that transfer well to downstream tasks like toxicity, solubility, and bioactivity prediction — with far less data than training from scratch.

In [ ]:
!pip install transformers datasets torch rdkit scikit-learn -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from rdkit import Chem

MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

# Test tokenization
test_smiles = "CC(=O)Oc1ccccc1C(=O)O"
tokens = tokenizer(test_smiles, return_tensors="pt")
print(f"\nAspirin SMILES: {test_smiles}")
print(f"Token IDs: {tokens['input_ids'][0][:10].tolist()}...")
print(f"N tokens: {tokens['input_ids'].shape[1]}")

## Prepare the dataset

In [ ]:
# Hepatotoxicity dataset (representative examples)
tox_data = [
    ("CC(=O)Nc1ccc(O)cc1", 0),       # Paracetamol - hepatotoxic at overdose
    ("CCCc1nc2ccccc2c(=O)n1C", 0),   # toxic compound
    ("O=S(=O)(O)c1cccc2cccc(S(=O)(=O)O)c12", 0),  # toxic
    ("CC(=O)Oc1ccccc1C(=O)O", 1),    # Aspirin - non-hepatotoxic
    ("CC(C)Cc1ccc(cc1)C(C)C(=O)O", 1), # Ibuprofen
    ("Cn1cnc2c1c(=O)n(C)c(=O)n2C", 1), # Caffeine
    ("NCCc1ccc(O)c(O)c1", 1),          # Dopamine
    ("CC(N)Cc1ccccc1", 1),             # Amphetamine
    ("OC(=O)c1ccccc1", 1),             # Benzoic acid
    ("CC(=O)c1ccc(cc1)C(C)(C)C", 1),   # p-tert-butylacetophenone
]

smiles_list = [s for s, _ in tox_data]
labels_list = [l for _, l in tox_data]

train_smi, test_smi, train_lbl, test_lbl = train_test_split(
    smiles_list, labels_list, test_size=0.2, stratify=labels_list, random_state=42
)
print(f"Train: {len(train_smi)}, Test: {len(test_smi)}")

def tokenize(smiles_list, labels, max_len=128):
    enc = tokenizer(smiles_list, truncation=True, padding=True,
                    max_length=max_len, return_tensors="pt")
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

train_enc = tokenize(train_smi, train_lbl)
test_enc  = tokenize(test_smi, test_lbl)

class MolDataset(torch.utils.data.Dataset):
    def __init__(self, enc):
        self.enc = enc
    def __len__(self): return len(self.enc["input_ids"])
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}

train_ds = MolDataset(train_enc)
test_ds  = MolDataset(test_enc)
print(f"Dataset ready: {len(train_ds)} train, {len(test_ds)} test")

## Fine-tune ChemBERTa

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    return {
        "accuracy": accuracy_score(labels, preds),
        "auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.5,
    }

args = TrainingArguments(
    output_dir="./chemberta_tox",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
results = trainer.evaluate()
print(f"\nFinal eval results: {results}")

## Zero-shot prediction on new compounds

In [ ]:
model.eval()
new_compounds = {
    "Chloroform":   "ClC(Cl)Cl",
    "Vitamin C":    "OC[C@H](O)[C@@H]1OC(=O)C(O)=C1O",
    "Metformin":    "CN(C)C(=N)NC(=N)N",
    "Acetaminophen":"CC(=O)Nc1ccc(O)cc1",
}

print("Toxicity predictions (0=toxic, 1=non-toxic):")
with torch.no_grad():
    for name, smi in new_compounds.items():
        mol = Chem.MolFromSmiles(smi)
        if mol is None: continue
        enc = tokenizer(smi, return_tensors="pt", truncation=True, max_length=128)
        logits = model(**enc).logits
        probs  = torch.softmax(logits, dim=-1)[0]
        pred_class = torch.argmax(probs).item()
        print(f"  {name:20s}: {'non-toxic' if pred_class==1 else 'toxic':12s}  confidence={probs[pred_class]:.3f}")

## Key takeaways
- ChemBERTa tokenizes SMILES as chemical 'words' — the model learns chemical grammar
- Fine-tuning needs very little data — transfer learning from 77M PubChem SMILES is powerful
- For production: use larger datasets from Tox21, SIDER, ClinTox, DILI (drug-induced liver injury)
- Compare against RF baseline (Tutorial 13) — if similar AUC, prefer the interpretable RF